# 03b. Preparação de Dados Multi-Estado e Recorrência
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Este notebook utiliza a lógica modularizada em `src/data/preprocessing.py` para transformar o dataset original SABI num formato de **Counting Process** compatível com modelos de sobrevivência avançados (PWP, RSF, DeepHit).

### Objetivos:
1. **Mapeamento de Estados:** 0 (Saudável), 1 (Distress), 2 (Falência).
2. **Lagging Temporal:** Prevenir *Data Leakage* (X em $t-1$ para prever $y$ em $t$).
3. **Análise de Transições:** Validar a dinâmica de recuperação e falência das PME portuguesas.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Adicionar a raiz do projeto ao sys.path para importar do src/
sys.path.append('..')
from src.data.preprocessing import prepare_multistate_data

DATA_INTERIM = "../data/interim"
DATA_PROCESSED = "../data/processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)

print("A carregar dados interim (com features)...\n"),
df_micro = pd.read_csv(os.path.join(DATA_INTERIM, "micro_features.csv"))
df_macro = pd.read_csv(os.path.join(DATA_INTERIM, "macro_consolidated.csv"))

A carregar dados interim (com features)...



## 1. Execução do Pré-processamento Multi-Estado

Aplicamos a função `prepare_multistate_data` que automatiza a limpeza, definição de estados e lagging.

In [2]:
df_prepared = prepare_multistate_data(df_micro)

print(f"Dataset preparado: {df_prepared.shape[0]} observações de {df_prepared['NIF Code'].nunique()} empresas.")
print("\nDistribuição de Estados (y):")
print(df_prepared['State'].value_counts(normalize=True).sort_index() * 100)

Dataset preparado: 74968 observações de 12639 empresas.

Distribuição de Estados (y):
State
0    78.721588
1    18.741330
2     2.537082
Name: proportion, dtype: float64


## 2. Matriz de Transição de Estados

Validamos a probabilidade de uma empresa mudar de estado entre o ano $t-1$ e $t$.

In [3]:
trans_matrix = pd.crosstab(df_prepared['State_Prev'], df_prepared['State'], normalize='index')
print("Matriz de Transição (Probabilidades):")
display(trans_matrix.style.format("{:.2%}").background_gradient(cmap='Blues'))

print("\nLegenda:\n0 -> Saudável | 1 -> Distress | 2 -> Falência")

Matriz de Transição (Probabilidades):


State,0,1,2
State_Prev,,,
0.000000,91.17%,5.72%,3.11%
1.000000,23.43%,76.57%,0.00%



Legenda:
0 -> Saudável | 1 -> Distress | 2 -> Falência


## 3. Integração com Dados Macroeconómicos

Combinamos os dados das empresas com os indicadores económicos nacionais (PIB, Desemprego, Juros).

In [4]:
df_macro.rename(columns={'year': 'Year'}, inplace=True)
df_final = pd.merge(df_prepared, df_macro, on='Year', how='left')

# Guardar o dataset final pronto para modelação
df_final.to_csv(os.path.join(DATA_PROCESSED, "final_multistate_dataset.csv"), index=False)

print(f"\nSucesso! Dataset Final guardado em: {os.path.join(DATA_PROCESSED, 'final_multistate_dataset.csv')}")
print(f"Colunas totais: {len(df_final.columns)}")


Sucesso! Dataset Final guardado em: ../data/processed\final_multistate_dataset.csv
Colunas totais: 75
